In [1]:
import pandas as pd
import warnings

In [2]:
df = pd.read_csv('./outputs/Global_Graphite_Projects.csv')
print(f"Loaded: {len(df)} rows, {len(df.columns)} columns")

change_log = []

def log_change(idx, name, field, old_val, new_val, source):
    change_log.append({
        'row_index': idx, 'name': name, 'field': field,
        'old_value': old_val, 'new_value': new_val, 'source': source
    })

Loaded: 183 rows, 90 columns


# 1. Data Imputation

## 1.1. China

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# SOURCE: USGS Minerals Yearbook 2023 - China [Advance Release], Feb 2026
# https://pubs.usgs.gov/myb/vol3/2023/myb3-2023-china.pdf
#
# Table 2 (p. 9.24): "Structure of the Mineral Industry in 2023"
#   - Lists 8 graphite facilities with capacities in THOUSAND metric tons
#   - This is the most current publicly available facility-level data for China
#
# Narrative text (p. 9.5–9.6): "Graphite" section under "Industrial Minerals"
#   - Names 3 additional large-scale facilities that came online in 2023
#   - These are NOT listed in Table 2 (they are new capacity additions)
#   - China's total natural graphite capacity reached 1.8 Mt/yr in 2023
#     (up 600,000 t/yr from 2022)
#   - Natural graphite production: 1.21 Mt in 2023 (up from 1.16 Mt in 2022)
#
# IMPORTANT: We focus only on NATURAL graphite. BTR also added 200,000 t/yr
# synthetic capacity - we exclude that.
# ══════════════════════════════════════════════════════════════════════════════

SOURCE = 'USGS Minerals Yearbook 2023 — China (Feb 2026), Table 2 (p. 9.24) & Graphite text (p. 9.5-9.6)'

In [7]:
# Update existing records from Table 2
# Match our existing China facilities to Table 2 entries and update capacities

updates = [
    # (search_keyword_in_Name_or_Operator, new_capacity_mt, new_operator, notes)
    
    # "Mine in Jixi" / Jixi Aoyu: our GDB has 60,000 (from 2019 MYB); Table 2 shows 100 kmt
    ('Aoyu',    100_000, 'Jixi Aoyu Graphite Co. Ltd.',
     'Updated from 60,000 (2019 MYB) to 100,000 (2023 MYB Table 2)'),
    
    # "Plant in Qingdao" / Hensen: our GDB has -999; Table 2 shows 30 kmt
    # Note: Our GDB labels this as "Qingdao Hensen" but Table 2 shows
    # Hensen is in Heilongjiang/Jiangsu/Nei Mongol, not Qingdao
    ('Hensen',  30_000,  'Hensen Graphite Co. Ltd.',
     'Was -999 (unknown); Table 2 shows 30 kmt. Location is Heilongjiang/Jiangsu/Nei Mongol, not Qingdao'),
    
    # "Liumao Mine": our GDB has 80,000; Table 2 (2023) now lists this as
    # "Jixi Puchen Graphite Co. Ltd." (same location, same capacity)
    ('Liumao',  80_000,  'Jixi Puchen Graphite Co. Ltd.',
     'Company renamed from Jixi Liumao (2022 MYB) to Jixi Puchen (2023 MYB). Capacity unchanged at 80 kmt'),
    
    # "Mine in Xinghe": capacity unchanged at 10,000
    ('Xinghe',  10_000,  'Nei Mongol Xinghe Jingyin Graphite Co. Ltd.',
     'Company name spelling updated: Jingxin (2022) → Jingyin (2023). Capacity unchanged'),
    
    # "Qingdao Mine" / Yanxin: capacity unchanged at 28,000
    ('Yanxin',  28_000,  'Qingdao Yanxin Graphite Products Co. Ltd.',
     'Capacity confirmed at 28 kmt in 2023 MYB Table 2'),
]

for keyword, new_cap, new_operator, notes in updates:
    mask = (
        (df['Country (Short Form)'] == 'China') &
        (df['Major Operating Company'].str.contains(keyword, case=False, na=False) |
         df['Facility Name'].str.contains(keyword, case=False, na=False))
    )
    
    matched = df[mask]
    if matched.empty:
        print(f"  ⚠ No match found for '{keyword}' — will add as new record")
        continue
    
    for idx in matched.index:
        old_cap = df.loc[idx, 'Annual Production Capacity']
        old_op = df.loc[idx, 'Major Operating Company']
        
        # Update capacity
        if pd.isna(old_cap) or old_cap < 0 or old_cap != new_cap:
            df.loc[idx, 'Annual Production Capacity'] = new_cap
            df.loc[idx, 'Capacity Units'] = 'mt'
            log_change(idx, df.loc[idx, 'Facility Name'], 'Annual Production Capacity',
                       old_cap, new_cap, SOURCE)
            print(f"  ✓ {df.loc[idx, 'Facility Name']}: capacity {old_cap} → {new_cap} mt")
        else:
            print(f"  - {df.loc[idx, 'Facility Name']}: capacity already correct ({new_cap} mt)")
        
        # Update operator name
        if new_operator and old_op != new_operator:
            df.loc[idx, 'Major Operating Company'] = new_operator
            log_change(idx, df.loc[idx, 'Facility Name'], 'Major Operating Company',
                       old_op, new_operator, SOURCE)
        
        # Update reference year
        df.loc[idx, 'Reference Year'] = 2023

print(f"\nUpdated {len([c for c in change_log if 'Capacity' in c['field']])} capacity values")

  ✓ Mine in Jixi: capacity 60.0 → 100000 mt
  ✓ Plant in Qingdao: capacity -999.0 → 30000 mt
  ✓ Liumao Mine: capacity 80.0 → 80000 mt
  ✓ Mine in Xinghe: capacity 10.0 → 10000 mt
  ✓ Qingdao Mine: capacity 28.0 → 28000 mt

Updated 5 capacity values


In [8]:
# Add new facilities from Table 2 not in our data
# These are in the official Table 2 but missing from our GDB

table2_new = [
    # (Name, Company, Location, Capacity_mt_or_None, Notes)
    ('Qingdao Haida Graphite',
     'Qingdao Haida Graphite Co. Ltd.',
     'Shandong, Qingdao, Pingdu',
     60_000,
     'Table 2 entry; 60 kmt capacity'),
    
    ('Heilongjiang Hegang Graphite Industry',
     'Heilongjiang Hegang Graphite Industry Co. Ltd.',
     'Heilongjiang, Luobei',
     None,  # NA in Table 2
     'Table 2 entry; capacity NA'),
    
    ('Luobei Haida Graphite',
     'Luobei Haida Graphite Co. Ltd.',
     'Heilongjiang, Hegang',
     None,  # NA in Table 2
     'Table 2 entry; capacity NA'),
]

added_t2 = 0
for name, company, location, cap, notes in table2_new:
    # Check not already present
    keyword = name.split()[0]
    already = df[
        (df['Country (Short Form)'] == 'China') &
        (df['Facility Name'].str.contains(keyword, case=False, na=False) |
         df['Major Operating Company'].str.contains(keyword, case=False, na=False))
    ]
    
    if not already.empty:
        print(f"  - '{name}' already matched by '{already.iloc[0]['Facility Name']}' — skipping")
        continue
    
    new_row = {
        'Region': 'China',
        'Source_Layer': 'CHN_MYB_2023_Table2',
        'Country (Short Form)': 'China',
        'Facility Name': name,
        'Commodity': 'Graphite',
        'Status': 'Assumed Active',
        'Facility Type': 'Mines and Quarries',
        'Annual Production Capacity': cap,
        'Capacity Units': 'mt' if cap else None,
        'Major Operating Company': company,
        'Location Description': location,
        'Reference Year': 2023,
        'Facility Comments': notes,
        'Data Source': SOURCE,
    }
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    log_change(len(df)-1, name, 'NEW RECORD', None, name, SOURCE)
    added_t2 += 1
    print(f"  + Added: {name} ({cap if cap else 'NA'} mt)")

print(f"\nAdded {added_t2} new facilities from Table 2")

  - 'Qingdao Haida Graphite' already matched by 'Plant in Qingdao' — skipping
  + Added: Heilongjiang Hegang Graphite Industry (NA mt)
  + Added: Luobei Haida Graphite (NA mt)

Added 2 new facilities from Table 2


C:\Users\157412\AppData\Local\Temp\ipykernel_33032\1177837915.py:55: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
C:\Users\157412\AppData\Local\Temp\ipykernel_33032\1177837915.py:55: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)


In [9]:
# Add new large facilities from narrative text (p. 9.5–9.6)
# These are major 2023 capacity additions described in the Graphite section
# but NOT listed as individual entries in Table 2

narrative_new = [
    # (Name, Company, Location, Capacity_mt, Stage, Notes)
    ('Hegang Natural Graphite Processing Facility',
     'China Minmetals Co. Ltd.',
     'Heilongjiang, Hegang',
     200_000,
     'Production',
     'Pilot production started late August 2023. '
     '200,000 t/yr nameplate capacity for natural graphite processing.'),
    
    ('BTR Heilongjiang Natural Graphite (Phase 1)',
     'BTR New Material Group Co. Ltd.',
     'Heilongjiang',
     50_000,
     'Production',
     'Phase 1: 50,000 t/yr natural graphite. '
     'Target: 200,000 t/yr natural + 400,000 t/yr synthetic (synthetic excluded). '
     'Only natural graphite capacity counted here.'),
    
    ('Jixi Northeast Graphite Flake Plant',
     'Jixi Northeast Mineral Resources Co. Ltd.',
     'Heilongjiang, Jixi',
     100_000,
     'Construction',
     'Construction of 100,000 t/yr graphite flake plant completed in 2023.'),
]

SOURCE_NARRATIVE = SOURCE.replace('Table 2 (p. 9.24) &', '')  # Just the narrative ref

added_narr = 0
for name, company, location, cap, stage, notes in narrative_new:
    # Check not already present using first distinctive word of company name
    company_kw = company.split()[0]  # 'China', 'BTR', 'Jixi'
    # For "China Minmetals" use "Minmetals"; for "Jixi Northeast" use "Northeast"
    if company_kw in ('China', 'Jixi'):
        company_kw = company.split()[1]
    
    already = df[
        (df['Country (Short Form)'] == 'China') &
        (df['Facility Name'].str.contains(company_kw, case=False, na=False) |
         df['Major Operating Company'].str.contains(company_kw, case=False, na=False))
    ]
    
    if not already.empty:
        print(f"  - '{name}' may overlap with '{already.iloc[0]['Facility Name']}' — skipping")
        continue
    
    new_row = {
        'Region': 'China',
        'Source_Layer': 'CHN_MYB_2023_Narrative',
        'Country (Short Form)': 'China',
        'Facility Name': name,
        'Commodity': 'Graphite',
        'Status': stage,
        'Facility Type': 'Mineral Processing Plants',
        'Annual Production Capacity': cap,
        'Capacity Units': 'mt',
        'Major Operating Company': company,
        'Location Description': location,
        'Reference Year': 2023,
        'Facility Comments': notes,
        'Data Source': SOURCE_NARRATIVE,
    }
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    log_change(len(df)-1, name, 'NEW RECORD', None, name, SOURCE_NARRATIVE)
    added_narr += 1
    print(f"  + Added: {name} ({cap:,} mt)")

print(f"\nAdded {added_narr} new facilities from narrative text")

  + Added: Hegang Natural Graphite Processing Facility (200,000 mt)
  + Added: BTR Heilongjiang Natural Graphite (Phase 1) (50,000 mt)
  + Added: Jixi Northeast Graphite Flake Plant (100,000 mt)

Added 3 new facilities from narrative text


In [14]:
df[df['Region'] == 'China']['Annual Production Capacity'].sum()

np.float64(598000.0)

In [15]:
# ── SUMMARY ──
change_df = pd.DataFrame(change_log)

print(f"\n{'='*70}")
print(f"CHINA DATA IMPUTATION SUMMARY")
print(f"{'='*70}")
print(f"Source: {SOURCE}")
print(f"Total field-level changes: {len(change_df)}")

# Show capacity changes
cap_changes = change_df[change_df['field'] == 'Annual Production Capacity']
if not cap_changes.empty:
    print(f"\nCapacity updates ({len(cap_changes)}):")
    for _, ch in cap_changes.iterrows():
        print(f"  {ch['name']}: {ch['old_value']} → {ch['new_value']} mt")

# Show new records
new_records = change_df[change_df['field'] == 'NEW RECORD']
if not new_records.empty:
    print(f"\nNew records added ({len(new_records)}):")
    for _, ch in new_records.iterrows():
        print(f"  {ch['new_value']}")

# China coverage summary
china = df[(df['Country (Short Form)'] == 'China') & (df['Annual Production Capacity'] > 0)]
total_cap = china['Annual Production Capacity'].sum()
print(f"\n--- China Coverage ---")
print(f"Named facilities with capacity: {len(china)}")
print(f"Total named capacity: {total_cap:,.0f} mt")
print(f"USGS national capacity (2023): 1,800,000 mt/yr")
print(f"Coverage: {100*total_cap/1_800_000:.0f}%")
print(f"Untracked (hundreds of small/medium mines): {1_800_000 - total_cap:,.0f} mt")

# Show all China facilities
print(f"\nAll China graphite facilities:")
china_all = df[df['Country (Short Form)'] == 'China'][
    ['Facility Name', 'Annual Production Capacity', 'Major Operating Company', 'Source_Layer']
]
print(china_all.to_string(index=False))


CHINA DATA IMPUTATION SUMMARY
Source: USGS Minerals Yearbook 2023 — China (Feb 2026), Table 2 (p. 9.24) & Graphite text (p. 9.5-9.6)
Total field-level changes: 15

Capacity updates (5):
  Mine in Jixi: 60.0 → 100000 mt
  Plant in Qingdao: -999.0 → 30000 mt
  Liumao Mine: 80.0 → 80000 mt
  Mine in Xinghe: 10.0 → 10000 mt
  Qingdao Mine: 28.0 → 28000 mt

New records added (5):
  Heilongjiang Hegang Graphite Industry
  Luobei Haida Graphite
  Hegang Natural Graphite Processing Facility
  BTR Heilongjiang Natural Graphite (Phase 1)
  Jixi Northeast Graphite Flake Plant

--- China Coverage ---
Named facilities with capacity: 8
Total named capacity: 598,000 mt
USGS national capacity (2023): 1,800,000 mt/yr
Coverage: 33%
Untracked (hundreds of small/medium mines): 1,202,000 mt

All China graphite facilities:
                              Facility Name  Annual Production Capacity                        Major Operating Company            Source_Layer
                                Liumao Mine

In [16]:
# SAVE
df.to_csv('./outputs/Global_Graphite_Projects_imputed.csv', index=False)
print(f"Saved: {len(df)} records to Global_Graphite_Projects_imputed.csv")

change_df.to_csv('./outputs/china_imputation_change_log.csv', index=False)
print(f"Saved: {len(change_df)} changes to china_imputation_change_log.csv")

Saved: 188 records to Global_Graphite_Projects_imputed.csv
Saved: 15 changes to china_imputation_change_log.csv
